# 转生千金 · 全角色日语成品配音一键生成（3-Cell 极简版）

**3 步拿成品**：① 在 Cell 1 填入 `FISH_API_KEY`（或在左侧 🔑 Secrets 配置） → ② 点击顶部菜单 **代码执行程序 → 全部运行 (Run All)** → ③ 自动下载 `ojousama_jp_voice_pack.zip`，解压覆盖游戏仓库根目录即可生效。

- **硬件全自动适配**：`H100 80GB` → `drama-3 (FP8)` ｜ `A100 80GB` → `drama-3-int4` ｜ `A100 40GB` → `drama-3-int4`（自动收紧显存 + 19s 参考音）｜ `L4 / T4 / CPU` → 免下权重直连 Fish Audio API（默认 `drama-3`，可改 `s2.1-pro-free`）。
- **全自动生产规格**：默认全量 `all`（1,276 条，可选 `main` 900 条）；64 kbps mono MP3、`-16 LUFS / -1.5 dBTP`、40ms/120ms 修剪、自动重抽与 Google Drive SHA-256 断点续传。

### ⚙️ Cell 1：唯一配置与环境一键就绪

In [ ]:
#@title ⚙️ 唯一配置与环境一键就绪
FISH_API_KEY = "" #@param {type:"string"}
SCOPE = "all" #@param ["all", "main"]
SAVE_TO_DRIVE = True #@param {type:"boolean"}

# L4 / T4 / CPU 自动切 Fish Audio API 时的模型：默认 "drama-3"；如需 $0 免费通道可改为 "s2.1-pro-free"
FISH_API_MODEL = "drama-3"

import getpass
import json
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/juanhanyu/fictional-adventure.git"
REPO_DIR = Path("/content/fictional-adventure")
CFG_PATH = Path("/content/voice_cfg.json")

# 1. 读取 FISH_API_KEY（表单 -> Colab Secrets -> 环境变量 -> 隐藏输入框；绝不落盘）
api_key = FISH_API_KEY.strip()
if not api_key:
    try:
        from google.colab import userdata
        api_key = (userdata.get("FISH_API_KEY") or "").strip()
        if api_key:
            print("✅ 已从 Colab Secrets 读取 FISH_API_KEY")
    except Exception:
        pass
if not api_key:
    api_key = os.environ.get("FISH_API_KEY", "").strip()
if not api_key:
    api_key = getpass.getpass("请输入 FISH_API_KEY（输入隐藏，仅存内存）: ").strip()
if not api_key:
    raise RuntimeError("未提供 FISH_API_KEY：请在 Cell 1 表单填写或在 Colab Secrets 配置 FISH_API_KEY 后重跑。")
os.environ["FISH_API_KEY"] = api_key

# 2. 挂载 Google Drive（取消授权或非 Colab 环境自动回退 /content/voice_work，不报错）
drive_ready = False
work_dir = Path("/content/voice_work")
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        if Path("/content/drive/MyDrive").is_dir():
            work_dir = Path("/content/drive/MyDrive/fictional_adventure_voice")
            drive_ready = True
            print(f"✅ Google Drive 已挂载，断点与成品缓存目录: {work_dir}")
    except Exception as e:
        print(f"ℹ️ Google Drive 未挂载，自动回退本地目录 {work_dir}（{e}）")
work_dir.mkdir(parents=True, exist_ok=True)
hf_cache = work_dir / "hf_cache"
hf_cache.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(hf_cache)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(hf_cache / "hub")

# 3. 自动探测 GPU 并锁定最优推理方案（零人工选择）
def detect_gpu():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
        if out:
            name, mem_mb, cc = [x.strip() for x in out.splitlines()[0].split(",")]
            major, minor = [int(x) for x in cc.split(".")[:2]]
            return {"has_gpu": True, "name": name, "vram_gb": round(float(mem_mb) / 1024.0, 1), "sm": major * 10 + minor}
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            p = torch.cuda.get_device_properties(0)
            return {"has_gpu": True, "name": p.name, "vram_gb": round(p.total_memory / (1024 ** 3), 1), "sm": p.major * 10 + p.minor}
    except Exception:
        pass
    return {"has_gpu": False, "name": "CPU", "vram_gb": 0.0, "sm": 0}

gpu = detect_gpu()
extra_serve_args = []
max_ref_seconds = 55
max_ref_clips = 2
concurrency = 2

if gpu["has_gpu"] and gpu["sm"] >= 90 and gpu["vram_gb"] >= 70.0:
    tier = "H100 80GB (sm_90 · FP8)"
    backend = "sglang"
    model_id = "fishaudio/drama-3"
elif gpu["has_gpu"] and gpu["sm"] >= 80 and gpu["vram_gb"] >= 70.0:
    tier = "A100 80GB (sm_80 · INT4)"
    backend = "sglang"
    model_id = "fishaudio/drama-3-int4"
elif gpu["has_gpu"] and gpu["sm"] >= 80 and gpu["vram_gb"] >= 36.0:
    tier = "A100 40GB (sm_80 · INT4 显存收紧)"
    backend = "sglang"
    model_id = "fishaudio/drama-3-int4"
    extra_serve_args = [
        "--tts_engine.gpu_memory_fraction", "0.58",
        "--vocoder.gpu_memory_fraction", "0.12",
        "--tts_engine.factory.max_new_tokens", "1024",
        "--preprocessing.total_reserve_bytes", "3GiB",
        "--preprocessing.factory.max_reference_audio_seconds", "20",
    ]
    max_ref_seconds = 19
    max_ref_clips = 1
    concurrency = 1
else:
    tier = f"{gpu['name']} ({gpu['vram_gb']} GB) → 自动切 Fish Audio API"
    backend = "fish_api"
    model_id = FISH_API_MODEL
    concurrency = 2

print(f"🖥️ 硬件检测: {gpu['name']} | {gpu['vram_gb']} GB | sm_{gpu['sm']} ==> {tier}")
if backend == "fish_api":
    print(f"⚡ 免下 22GB 权重，直连 Fish Audio API（模型: {FISH_API_MODEL}；如需 $0 免费通道可一键改 FISH_API_MODEL = \"s2.1-pro-free\"）")

# 4. 克隆 / 同步仓库
def run(cmd, cwd=None, check=True):
    print(f"$ {' '.join(cmd)}", flush=True)
    proc = subprocess.run(cmd, cwd=cwd)
    if check and proc.returncode != 0:
        raise RuntimeError(f"命令失败 (exit={proc.returncode}): {' '.join(cmd)}")
    return proc.returncode

if not (REPO_DIR / ".git").exists():
    run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)])
else:
    run(["git", "fetch", "origin", "main", "--depth", "1"], cwd=str(REPO_DIR))
    run(["git", "checkout", "main"], cwd=str(REPO_DIR))
    run(["git", "reset", "--hard", "origin/main"], cwd=str(REPO_DIR))

# 5. 安装 FFmpeg 及最小依赖（仅本地 A100/H100 安装 sglang-omni[fishaudio]）
if not shutil.which("ffmpeg"):
    run(["apt-get", "-qq", "update"])
    run(["apt-get", "-qq", "-y", "install", "ffmpeg"])
else:
    print("✅ ffmpeg 已就绪")

if backend == "sglang":
    run(["pip", "install", "-q", "-U", "uv", "requests", "huggingface_hub"])
    print("🚀 安装 sglang-omni[fishaudio]（首次约 1–3 分钟）...", flush=True)
    rc = run(["uv", "pip", "install", "--system", "--prerelease=allow", "sglang-omni[fishaudio]"], check=False)
    if rc != 0:
        sgl_src = Path("/content/sglang-omni")
        if not sgl_src.exists():
            run(["git", "clone", "--depth", "1", "https://github.com/sgl-project/sglang-omni.git", str(sgl_src)])
        run(["uv", "pip", "install", "--system", "--prerelease=allow", "-e", ".[fishaudio]"], cwd=str(sgl_src))
else:
    run(["pip", "install", "-q", "requests"])

# 6. 检查 Node.js（用于 Cell 3 生成清单；缺失或 <18 时自动走 Python 等价兜底）
node_ok = False
if shutil.which("node"):
    try:
        nver = subprocess.check_output(["node", "-v"], text=True).strip()
        node_ok = int(nver.lstrip("v").split(".")[0]) >= 18
        print(f"{'✅' if node_ok else 'ℹ️'} Node.js: {nver}" + ("" if node_ok else "（<18，Cell 3 自动走 Python 兜底）"))
    except Exception:
        print("ℹ️ Node.js 版本检测异常，Cell 3 自动走 Python 兜底")
else:
    print("ℹ️ 未安装 Node.js，Cell 3 将自动用 Python 等价生成清单")

# 7. 写入运行配置（绝不含 API Key）
cfg = {
    "scope": SCOPE,
    "backend": backend,
    "model_id": model_id,
    "fish_api_model": FISH_API_MODEL,
    "tier": tier,
    "extra_serve_args": extra_serve_args,
    "max_ref_seconds": max_ref_seconds,
    "max_ref_clips": max_ref_clips,
    "concurrency": concurrency,
    "drive_ready": drive_ready,
    "work_dir": str(work_dir),
    "repo_dir": str(REPO_DIR),
    "node_ok": node_ok,
}
CFG_PATH.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n✅ 环境与配置已就绪 | scope={SCOPE} | backend={backend} | model={model_id}")


### 🎙️ Cell 2：一键生成全部成品日配（拉取 25 角色样本 → 启动 Drama 3 → 全量合成与质检归一化）

In [ ]:
#@title 🎙️ 一键生成全部成品日配（拉取 25 角色样本 → 启动 Drama 3 → 全量合成与质检归一化）
import json
import os
import signal
import subprocess
import time
import urllib.request
from pathlib import Path

CFG_PATH = Path("/content/voice_cfg.json")
if not CFG_PATH.is_file():
    raise FileNotFoundError("未找到配置文件，请先运行 Cell 1。")

cfg = json.loads(CFG_PATH.read_text(encoding="utf-8"))
repo_dir = Path(cfg["repo_dir"])
work_dir = Path(cfg["work_dir"])
voice_out_dir = work_dir / "assets_voice"
cache_file = work_dir / ".synth_cache.json"
refs_dir = work_dir / "refs_cache"
refs_resolved = work_dir / "refs_resolved.json"
qa_report_path = repo_dir / "tools/voice/qa_report.json"
serve_log_path = Path("/content/sgl_omni_serve.log")
SERVE_PORT = 8000
STARTUP_TIMEOUT_SEC = 900

voice_out_dir.mkdir(parents=True, exist_ok=True)
refs_dir.mkdir(parents=True, exist_ok=True)

# 单独重跑 Cell 2 时自动兜底恢复 Secrets 中的 Key
if not os.environ.get("FISH_API_KEY"):
    try:
        from google.colab import userdata
        k = (userdata.get("FISH_API_KEY") or "").strip()
        if k:
            os.environ["FISH_API_KEY"] = k
    except Exception:
        pass

def check_health(port: int) -> bool:
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=3) as resp:
            return getattr(resp, "status", 200) == 200
    except Exception:
        return False

def tail_log(path: Path, n: int = 40) -> str:
    if not path.is_file():
        return "(无日志)"
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    return "\n".join(lines[-n:]) if lines else "(日志为空)"

def stop_server(proc) -> None:
    if proc is None or proc.poll() is not None:
        return
    print("🧹 正在优雅关闭后台 sgl-omni 释放显存...", flush=True)
    try:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
    except Exception:
        proc.terminate()
    try:
        proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        except Exception:
            proc.kill()
        proc.wait(timeout=5)
    print("✅ sgl-omni 已关闭，显存已释放。")

serve_proc = None
serve_log_fp = None

try:
    if cfg["backend"] == "sglang":
        # 1. 拉取全部 25 套 fish.audio 参考样本并转 44.1 kHz WAV（自动降级 backup，支持断点复用）
        fetch_cmd = [
            "python3", "tools/voice/fetch_fish_refs.py",
            "--refs-dir", str(refs_dir),
            "--out", str(refs_resolved),
            "--max-seconds", str(cfg["max_ref_seconds"]),
            "--max-clips", str(cfg["max_ref_clips"]),
        ]
        custom_refs_dir = work_dir / "custom_refs"
        if custom_refs_dir.is_dir():
            fetch_cmd += ["--custom-refs-dir", str(custom_refs_dir)]
        print(f"📥 [1/3] 拉取 25 角色参考音:\n   {' '.join(fetch_cmd)}", flush=True)
        f_proc = subprocess.run(fetch_cmd, cwd=str(repo_dir))
        if not refs_resolved.is_file():
            raise RuntimeError("参考音拉取失败：未生成 refs_resolved.json，请检查 FISH_API_KEY。")
        r_summary = json.loads(refs_resolved.read_text(encoding="utf-8")).get("summary", {})
        if r_summary.get("resolved", 0) == 0:
            raise RuntimeError("所有 25 角色参考音均拉取失败，请核对 FISH_API_KEY 与网络连接。")
        if f_proc.returncode != 0:
            print(f"⚠️ 参考音拉取完成（失败 {r_summary.get('failed', 0)} 套），继续合成其余角色。", flush=True)

        # 2. 预下载权重并后台启动 sgl-omni serve，等待 /health 就绪
        if check_health(SERVE_PORT):
            print(f"✅ [2/3] 检测到 sgl-omni (:{SERVE_PORT}) 已就绪，直接复用。")
        else:
            print(f"📦 [2/3] 核对/下载模型权重: {cfg['model_id']} ...", flush=True)
            from huggingface_hub import snapshot_download
            snapshot_download(repo_id=cfg["model_id"])

            serve_cmd = [
                "sgl-omni", "serve",
                "--model-path", cfg["model_id"],
                "--port", str(SERVE_PORT),
                "--allowed-local-media-path", str(refs_dir.resolve()),
            ] + list(cfg.get("extra_serve_args") or [])
            print(f"🚀 后台启动 Drama 3 服务:\n   {' '.join(serve_cmd)}", flush=True)
            serve_log_fp = serve_log_path.open("w", encoding="utf-8")
            serve_proc = subprocess.Popen(
                serve_cmd,
                stdout=serve_log_fp,
                stderr=subprocess.STDOUT,
                cwd=str(repo_dir),
                env=os.environ.copy(),
                start_new_session=True,
            )

            t0 = time.monotonic()
            last_line = ""
            while time.monotonic() - t0 < STARTUP_TIMEOUT_SEC:
                if check_health(SERVE_PORT):
                    print(f"✅ sgl-omni serve 就绪（耗时 {int(time.monotonic() - t0)}s）", flush=True)
                    break
                if serve_proc.poll() is not None:
                    raise RuntimeError(
                        f"sgl-omni serve 异常退出 (code={serve_proc.returncode})，最后 40 行日志：\n"
                        f"{tail_log(serve_log_path, 40)}"
                    )
                cur = serve_log_path.read_text(encoding="utf-8", errors="replace").splitlines() if serve_log_path.is_file() else []
                tail = cur[-1] if cur else ""
                if tail and tail != last_line:
                    print(f"  [{int(time.monotonic() - t0):3d}s] {tail[:140]}", flush=True)
                    last_line = tail
                time.sleep(5)
            else:
                raise TimeoutError(
                    f"等待 sgl-omni /health 超时 ({STARTUP_TIMEOUT_SEC}s)，最后 40 行日志：\n"
                    f"{tail_log(serve_log_path, 40)}"
                )
    else:
        print(f"⚡ Fish Audio API 模式（{cfg['fish_api_model']}），直连云端合成，跳过本地权重与参考音拉取。", flush=True)

    # 3. 批量合成 + 时长/削波自动重抽 + 静音修剪与 -16 LUFS 归一化（SHA-256 断点直存 work_dir）
    synth_cmd = [
        "python3", "tools/voice/batch_synthesize.py",
        "--backend", cfg["backend"],
        "--scope", cfg["scope"],
        "--model-id", cfg["model_id"],
        "--out-dir", str(voice_out_dir),
        "--cache-file", str(cache_file),
        "--qa-report", str(qa_report_path),
        "--max-ref-clips", str(cfg["max_ref_clips"]),
        "--concurrency", str(cfg["concurrency"]),
        "--max-retries", "3",
        "--skip-manifest",
    ]
    if cfg["backend"] == "sglang":
        synth_cmd += [
            "--refs", str(refs_resolved),
            "--sglang-url", f"http://127.0.0.1:{SERVE_PORT}/v1/audio/speech",
        ]
    else:
        synth_cmd += ["--fish-api-model", cfg["fish_api_model"]]

    print(f"\n🎙️ [3/3] 开始全自动批量合成 (scope={cfg['scope']}):\n   {' '.join(synth_cmd)}", flush=True)
    s_proc = subprocess.run(synth_cmd, cwd=str(repo_dir))
    if qa_report_path.is_file():
        (work_dir / "qa_report.json").write_bytes(qa_report_path.read_bytes())
    elif s_proc.returncode != 0:
        raise RuntimeError(f"批量合成异常中止 (exit={s_proc.returncode})，请检查上方报错信息。")

    if s_proc.returncode != 0:
        print("⚠️ 批产完成但存在少量失败条目（其余成品已完整落盘，Cell 3 将继续打包）。")
    else:
        print("\n🎉 全部日配合成、质检与响度归一化完成！")
finally:
    stop_server(serve_proc)
    if serve_log_fp is not None:
        serve_log_fp.close()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


### 📦 Cell 3：自动生成清单并打包下载成品（`ojousama_jp_voice_pack.zip`）

In [ ]:
#@title 📦 自动生成清单并打包下载成品（ojousama_jp_voice_pack.zip）
from datetime import datetime, timezone
import json
import shutil
import subprocess
import zipfile
from pathlib import Path

CFG_PATH = Path("/content/voice_cfg.json")
if not CFG_PATH.is_file():
    raise FileNotFoundError("未找到配置文件，请先运行 Cell 1 与 Cell 2。")

cfg = json.loads(CFG_PATH.read_text(encoding="utf-8"))
repo_dir = Path(cfg["repo_dir"])
work_dir = Path(cfg["work_dir"])
src_voice_dir = work_dir / "assets_voice"
dst_voice_dir = repo_dir / "assets/voice"
qa_path = repo_dir / "tools/voice/qa_report.json"
if not qa_path.is_file() and (work_dir / "qa_report.json").is_file():
    shutil.copy2(work_dir / "qa_report.json", qa_path)

# 1. 将持久化目录中的成品 MP3 同步到仓库 assets/voice/（跳过同体积已存在文件）
dst_voice_dir.mkdir(parents=True, exist_ok=True)
if src_voice_dir.is_dir() and src_voice_dir.resolve() != dst_voice_dir.resolve():
    for src in src_voice_dir.rglob("*.mp3"):
        if not src.is_file() or src.name.endswith(".tmp.mp3") or src.stat().st_size <= 0:
            continue
        dst = dst_voice_dir / src.relative_to(src_voice_dir)
        if not dst.is_file() or dst.stat().st_size != src.stat().st_size:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)

# 2. 构建 js/voice-manifest.js 与 assets/voice/manifest.json（优先 Node >= 18，缺失则用 Python 等价兜底）
def build_manifest_py(root: Path) -> None:
    casting = json.loads((root / "tools/voice/casting_fish_audio.json").read_text(encoding="utf-8"))
    raw = json.loads((root / "tools/voice/script_jp_master.json").read_text(encoding="utf-8"))
    items = raw if isinstance(raw, list) else (raw.get("lines") or raw.get("entries") or raw.get("items") or [])
    vdir = root / "assets/voice"
    mp3s = [p for p in sorted(vdir.rglob("*.mp3")) if p.is_file() and not p.name.endswith(".tmp.mp3") and p.stat().st_size > 0]
    by_rel = {p.relative_to(root).as_posix(): p.stat().st_size for p in mp3s}
    by_id = {p.stem: (p.relative_to(root).as_posix(), p.stat().st_size) for p in mp3s}
    files, entries, by_tier, by_prof, by_char, seen = {}, {}, {}, {}, {}, set()
    total_bytes, missing = 0, 0
    for it in items:
        vid = it.get("voiceId")
        if not vid or vid in seen:
            continue
        seen.add(vid)
        exp = (it.get("audioPath") or (f"assets/voice/{str(it.get('audioRelPath', '')).lstrip('/')}" if it.get("audioRelPath") else "")).replace("\\", "/")
        hit = (exp, by_rel[exp]) if exp in by_rel else by_id.get(vid)
        if not hit:
            missing += 1
            continue
        rel, sz = hit
        cid, tier = it.get("charId"), it.get("tier")
        pid = it.get("profileId") or (casting.get("charIdToProfileId") or {}).get(cid or "")
        files[vid] = rel
        rec = {"path": rel, "charId": cid, "profileId": pid, "tier": tier, **{k: it[k] for k in ("sceneId", "kind", "cueKey") if it.get(k)}, "bytes": sz}
        entries[vid] = rec
        total_bytes += sz
        by_tier[tier or "unknown"] = by_tier.get(tier or "unknown", 0) + 1
        by_prof[pid or "unknown"] = by_prof.get(pid or "unknown", 0) + 1
        by_char[cid or "unknown"] = by_char.get(cid or "unknown", 0) + 1
    s_files, disk_bytes = dict(sorted(files.items())), sum(by_rel.values())
    now = datetime.now(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")
    j_data = {
        "schemaVersion": "1.0.0", "generatedAt": now, "sourceMode": "script_jp_master",
        "stats": {
            "masterEntryCount": len(items), "diskMp3Count": len(mp3s), "manifestCount": len(s_files), "brokenLinks": 0,
            "missingMp3Count": missing, "orphanMp3Count": sum(1 for p in mp3s if p.stem not in seen),
            "totalBytes": total_bytes, "totalMB": round(total_bytes / 1048576, 3),
            "diskTotalBytes": disk_bytes, "diskTotalMB": round(disk_bytes / 1048576, 3),
            "budgetMB": 120, "budgetUsedPct": round(disk_bytes / (120 * 1048576) * 100, 2),
            "byTier": by_tier, "byProfile": by_prof, "byChar": by_char,
        },
        "files": s_files, "entries": {k: entries[k] for k in s_files},
    }
    b_data = {
        "version": 1, "format": "mp3", "basePath": "assets/voice/", "generatedAt": now,
        "totalFiles": len(s_files), "totalBytes": total_bytes,
        "entries": {k: (v[13:] if v.startswith("assets/voice/") else v) for k, v in s_files.items()},
        "byChar": by_char, "speakerAliases": casting.get("speakerAliases") or {},
    }
    (vdir / "manifest.json").write_text(json.dumps(j_data, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    (root / "js/voice-manifest.js").write_text(
        "/* Auto-generated by tools/voice/build_manifest.mjs — DO NOT EDIT MANUALLY */\n"
        "(function (root) {\n  'use strict';\n"
        f"  root.VOICE_MANIFEST = {json.dumps(b_data, ensure_ascii=False, indent=2)};\n"
        "})(typeof globalThis !== 'undefined' ? globalThis : (typeof window !== 'undefined' ? window : this));\n",
        encoding="utf-8",
    )

manifest_built = False
if cfg.get("node_ok") and shutil.which("node"):
    m_proc = subprocess.run(["node", "tools/voice/build_manifest.mjs"], cwd=str(repo_dir))
    manifest_built = (m_proc.returncode == 0)
if not manifest_built:
    build_manifest_py(repo_dir)

# 3. 打包 assets/voice/ + js/voice-manifest.js + tools/voice/qa_report.json
zip_path = Path("/content/ojousama_jp_voice_pack.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(dst_voice_dir.rglob("*")):
        if not p.is_file() or p.name.endswith(".tmp.mp3"):
            continue
        if p.suffix.lower() == ".mp3" or p.name == "manifest.json":
            comp = zipfile.ZIP_STORED if p.suffix.lower() == ".mp3" else zipfile.ZIP_DEFLATED
            zf.write(p, p.relative_to(repo_dir).as_posix(), compress_type=comp)
    js_manifest = repo_dir / "js/voice-manifest.js"
    if js_manifest.is_file():
        zf.write(js_manifest, "js/voice-manifest.js", compress_type=zipfile.ZIP_DEFLATED)
    if qa_path.is_file():
        zf.write(qa_path, "tools/voice/qa_report.json", compress_type=zipfile.ZIP_DEFLATED)

# 4. 自动备份到 Google Drive 并触发浏览器下载
drive_zip = None
if Path("/content/drive/MyDrive").is_dir():
    drive_zip = Path("/content/drive/MyDrive/ojousama_jp_voice_pack.zip")
    shutil.copy2(zip_path, drive_zip)
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass

# 5. 三行极简汇总
manifest_json = dst_voice_dir / "manifest.json"
gen_count = json.loads(manifest_json.read_text(encoding="utf-8")).get("stats", {}).get("manifestCount", 0) if manifest_json.is_file() else 0
fail_count = json.loads(qa_path.read_text(encoding="utf-8")).get("stats", {}).get("failed", 0) if qa_path.is_file() else 0
zip_mb = zip_path.stat().st_size / 1048576

print(f"生成条数: {gen_count} 条 | 失败条数: {fail_count} 条")
print(f"成品压缩包: {zip_path.name} ({zip_mb:.2f} MB)" + (f" | 已同步 Google Drive: {drive_zip}" if drive_zip else ""))
print("解压提示: 将 ojousama_jp_voice_pack.zip 解压覆盖进游戏仓库根目录即可生效。")
